The classification model has a problem with pointing its attention to the ROI in the accompanying mask.  
If a photo has a lot of trash, the model is basically saying "yes" trash. if not it says other.  
This needs to be fixed with two things.  
- First im creating examples where the model is strongly disagreeing with me
- Im going to assign more weight to those examples

The other useful thing would be a pretrained model which knows how to point. I'll train the sample architecture on the shrinked TACO dataset.   
The model would need a size of photo to work with, we going with 224, easy.  

First though, I need to go through the TACO dset to see if I can extract something useful out of it.  

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import cv2
import random
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import cv2
import random
import torch
from mtrain.disk import DiskImage, DiskBooleanMask
from mtrain.utils import mkdir
from sklearn.model_selection import train_test_split
from tqdm import tqdm
import shutil
from mtrain.smallnet.unet.extract.draw import overlay_mask_on_img
from mtrain.utils import show
from fastai.data.core import DataLoaders, default_device
from collections import defaultdict
from torchvision import tv_tensors
from torchvision.transforms import v2
from PIL import Image
from fastai.vision.all import (
    vision_learner,
    mobilenet_v3_small,
    mobilenet_v3_large,
    accuracy,
    F1Score,
    CrossEntropyLossFlat,
    ProgressCallback,
)

In [ ]:
from mtrain.smallnet.unet.extract.query_taco import *
from pycocotools.coco import COCO

TACO = Path("/Users/hariomnarang/Desktop/personal/TACO/data")
ANN_FILE = TACO / "annotations.json"
coco = COCO(ANN_FILE)

In [ ]:
def get_image_to_cats(coco: COCO) -> dict[int, set[str]]:
    """
    Returns a list of image info dicts for images that contain annotations
    from more than one category.
    """
    img_to_cats: dict[int, set[str]] = {}
    for ann in coco.dataset["annotations"]:
        img_id = ann["image_id"]
        cat_id = ann["category_id"]
        cat_name = coco.loadCats([cat_id])[0]["supercategory"]
        img_to_cats.setdefault(img_id, set()).add(cat_name)
    return img_to_cats


def get_cat_to_cat_counts(img_to_cats: dict[int, set[str]]):
    res = defaultdict(lambda: defaultdict(lambda: 0))
    for cats in img_to_cats.values():
        for cat in cats:
            for other_cat in cats:
                if cat == other_cat:
                    continue
                else:
                    res[cat][other_cat] += 1
    return res 

# Generate dataset on dir

In [ ]:
from mtrain.smallnet.unet.extract.draw import overlay_mask_on_img
from mtrain.neg_mask.taco.build_dataset import iter_taco_samples, iter_taco_pairs


iterator = iter_taco_pairs(coco, TACO, 1024, 220, 20, 130)

In [ ]:
import itertools
pair = next(iterator)
to_show = list(itertools.chain.from_iterable(pair.pairs))
show(to_show)

In [ ]:
from mtrain.smallnet.unet.extract.draw import overlay_mask_on_img
from mtrain.neg_mask.taco.build_dataset import iter_taco_samples

# grab a few samples and visualise them
samples = []
for s in iter_taco_samples(coco, TACO, size=130):
    samples.append(s)
    if len(samples) == 6 * 4:
        break

# print(f"cat_names: {[s.cat_name for s in samples]}")

fig, axes = plt.subplots(len(samples) // 4, 4, figsize=(20, 20))
axes = axes.flatten()
for i, s in enumerate(samples):
    overlaid = overlay_mask_on_img(s.image, s.mask.astype(bool))
    axes[i].imshow(overlaid)
    axes[i].set_title(s.cat_name, fontsize=8)
    axes[i].set_title(s.cat_name, fontsize=8)
# axes[0, 0].set_ylabel("image")
# axes[1, 0].set_ylabel("mask")
plt.tight_layout()
plt.show()

In [ ]:
from mtrain.neg_mask.taco.build_dataset import build_taco_classification_dataset

In [ ]:
out_dir = Path(
    "/Users/hariomnarang/Desktop/personal/roads/datasets/test-samples/neg-masking/V1/taco"
)
import shutil

shutil.rmtree(out_dir, ignore_errors=True)
build_taco_classification_dataset(coco, TACO, out_dir, 1024, 220, 20, 130)

In [ ]:
def show_single_label(out_dir_root, label):
    sample_dirs = [d for d in (OUT_DIR / label).glob("*") if d.is_dir() and (d / "image.jpg").exists()]
    print(f"Total samples: {len(sample_dirs)}")
    print(f"Categories: {sorted({d.parent.name for d in sample_dirs})}")
    print(sample_dirs)

    chosen = random.sample(sample_dirs, 4 * 3)

    fig, axes = plt.subplots(len(chosen) // 4, 4, figsize=(20, 20))
    axes = axes.flatten()
    for i, d in enumerate(chosen):
        img = DiskImage.load(d / "image.jpg")
        mask = DiskBooleanMask.load(d / "mask.png")
        overlaid = overlay_mask_on_img(img, mask.astype(bool))
        axes[i].imshow(overlaid)
        axes[i].set_title(d.parent.name, fontsize=8)
        axes[i].set_axis_off()
    # axes[0, 0].set_ylabel("image")
    # axes[1, 0].set_ylabel("mask")
    plt.tight_layout()
    plt.show()

In [ ]:
show_single_label(OUT_DIR, "Lid")

In [ ]:
from mtrain.neg_mask.taco.extract import extract_mask_for_image_id
def show_category_matches(coco, taco_dir, label, img_to_cats, extra_labels, n_per_label=2):
    extra_label_by_ids = {}
    for lbl in extra_labels:
        required = set([lbl, label])
        img_ids = [img_id for img_id, cats in img_to_cats.items() if required.issubset(cats)]
        extra_label_by_ids[lbl] = img_ids

    print(len(extra_labels),n_per_label)
    _, axs = plt.subplots(len(extra_labels), n_per_label, figsize=(30,30)) 
    for i, lbl in enumerate(extra_labels):
        img_ids = extra_label_by_ids[lbl][:n_per_label]
        for j, img_id in enumerate(img_ids):
            image, mask = extract_mask_for_image_id(img_id, coco, taco_dir)
            mask = mask.astype(bool)
            overlaid = overlay_mask_on_img(image, mask)
            axs[i][j].imshow(overlaid)
            axs[i][j].set_title(f"{label}:{lbl}")
    plt.tight_layout()
    plt.show() 

In [ ]:
img_to_cats = get_image_to_cats(coco)
cat_to_cat_counts = get_cat_to_cat_counts(img_to_cats)

In [ ]:
CATS_TO_INCLUDE = list(cat_to_cat_counts.keys())

## Visualize

In [ ]:
import pandas as pd
# CATS_TO_INCLUDE = [
#     "Plastic bag & wrapper",
#     "Paper",
#     "Can",
#     "Cup",
#     "Plastic container",
#     "Other plastic",
#     "Unlabeled litter",
#     "Styrofoam piece",
#     "Bottle",
#     "Lid",
# ]

CATS_TO_INCLUDE = [
    "Bottle cap",
    "Pop tab",
    "Cigarette",
    "Straw",
    # "Rope & strings",
    # "Scrap metal",
    # "Broken glass",
    "Other plastic",
    "Unlabeled litter",
]


# cats = sorted(list(res.keys()))
img_to_cats = get_image_to_cats(coco)
cat_to_cat_counts = get_cat_to_cat_counts(img_to_cats)
df = pd.DataFrame(
    [[cat_to_cat_counts[r][c] for c in CATS_TO_INCLUDE] for r in CATS_TO_INCLUDE],
    index=CATS_TO_INCLUDE,
    columns=CATS_TO_INCLUDE,
)
df

In [ ]:
import cv2
from mtrain.disk import DiskImage, DiskBooleanMask

OUT_DIR = Path(
    "/Users/hariomnarang/Desktop/personal/roads/datasets/test-samples/neg-masking/V1/taco"
)

sample_dirs = []
for cat in CATS_TO_INCLUDE:
    dirs = [d for d in (OUT_DIR / cat).glob("*") if (d / "img_pair_0.jpg").exists()]
    sample_dirs.extend(dirs)
    
# sample_dirs = [d for d in OUT_DIR.rglob("*/") if (d / "img_pair_0.jpg").exists()]
print(f"Total samples: {len(sample_dirs)}")
print(f"Categories: {sorted({d.parent.name for d in sample_dirs})}")

chosen = random.sample(sample_dirs, 4 * 6)

fig, axes = plt.subplots(len(chosen) // 4, 4, figsize=(20, 20))
axes = axes.flatten()
for i, d in enumerate(chosen):
    img = DiskImage.load(d / "img_pair_0.jpg")
    mask = DiskBooleanMask.load(d / "mask_pair_0.png")
    overlaid = overlay_mask_on_img(img, mask.astype(bool), 1)
    axes[i].imshow(overlaid)
    axes[i].set_title(d.parent.name, fontsize=8)
    axes[i].set_axis_off()
# axes[0, 0].set_ylabel("image")
# axes[1, 0].set_ylabel("mask")
plt.tight_layout()
plt.show()

# fig, axes = plt.subplots(2, len(chosen), figsize=(3 * len(chosen), 6))
# for i, d in enumerate(chosen):
#     img = cv2.cvtColor(cv2.imread(str(d / "image.jpg")), cv2.COLOR_BGR2RGB)
#     mask = cv2.imread(str(d / "mask.png"), cv2.IMREAD_GRAYSCALE)
#     axes[0, i].imshow(img)
#     axes[0, i].set_title(d.parent.name, fontsize=7)
#     axes[0, i].axis("off")
#     axes[1, i].imshow(mask, cmap="gray")
#     axes[1, i].axis("off")
# axes[0, 0].set_ylabel("image")
# axes[1, 0].set_ylabel("mask")
# plt.tight_layout()
# plt.show()

# Model dataset and training

In [ ]:
DS_DIR = Path(
    "/Users/hariomnarang/Desktop/personal/roads/datasets/test-samples/neg-masking/V1/taco"
)

OUT_DIR = Path(
    "/Users/hariomnarang/Desktop/personal/roads/datasets/test-samples/neg-masking/V1/taco"
)

In [ ]:
CATS_TO_INCLUDE = [
    "Bottle cap",
    "Pop tab",
    "Cigarette",
    "Straw",
    # "Rope & strings",
    # "Scrap metal",
    # "Broken glass",
    "Other plastic",
    "Unlabeled litter",
]

LABELS = ["Empty"] + CATS_TO_INCLUDE


## Generic dataset to cache

In [ ]:
AREA_THRES = 5


def _is_valid_dir(direc: Path):
    valid = (
        direc.is_dir()
        and (direc / "img_pair_0.jpg").exists()
        and (direc / "mask_pair_0.png").exists()
    )
    return valid


def get_ds_dirs(ds_root, labels):
    res = []
    for label in labels:
        cls_root = ds_root / label
        print(cls_root.name)
        if not cls_root.is_dir():
            continue
        for d in cls_root.glob("*"):
            if _is_valid_dir(d):
                res.append(d)
    random.shuffle(res)
    return res


def _label_func(d: Path):
    return Path(d).parent.name


def _validate_labels_in_dirs(dirs, label_by_idx):
    for d in dirs:
        label = _label_func(d)
        if label not in label_by_idx:
            raise Exception(
                f"Label={label} not found for directory={d}. label_by_index={label_by_idx}"
            )


# def get_area(d):
#     return np.array(Image.open(d / "mask.png").convert("L")).sum()


total_dirs = get_ds_dirs(DS_DIR, LABELS)
dirs = total_dirs
# areas_and_dirs = [(get_area(d), d) for d in total_dirs]
# dirs = [d for (a, d) in areas_and_dirs if a > AREA_THRES or a == 0]

print(f"total directories scanned: {len(total_dirs)}")
print(f"filtered directories: {len(dirs)}")


labels = [_label_func(d) for d in dirs]
train_dirs, valid_dirs = train_test_split(
    dirs, test_size=0.2, stratify=labels, random_state=42
)

len(train_dirs), len(valid_dirs)

In [ ]:
from mtrain.neg_mask.model.dataset import persist_dataset_to_cache, GenericMaskClassificationDataset

def get_crops(d: Path):
    res = []
    for i in range(3):
        img = DiskImage.load(d / f"img_pair_{i}.jpg")
        mask = DiskBooleanMask.load(d / f"mask_pair_{i}.png")
        res.append((img, mask))
    return res

In [ ]:
train_ds = GenericMaskClassificationDataset(train_dirs, LABELS, True, 3, get_crops)
valid_ds = GenericMaskClassificationDataset(valid_dirs, LABELS, False, 3, get_crops)

In [ ]:
import shutil

def persist_cache(train_ds, valid_ds, cache_path):
    shutil.rmtree(cache_path, True)
    cache_path = mkdir(cache_path)
    persist_dataset_to_cache(train_ds, cache_path / "train")
    persist_dataset_to_cache(valid_ds, cache_path / "valid")

In [ ]:
CACHE_PATH = Path(
    "/Users/hariomnarang/Desktop/personal/roads/datasets/test-samples/neg-masking/V1/cache_taco"
)
persist_cache(train_ds, valid_ds, CACHE_PATH)

In [ ]:
train_ds.num_in_channels

## Load cached dataset

In [ ]:
from mtrain.neg_mask.model.dataset import DirectTensorLoadFromCacheDataset

train_ds = DirectTensorLoadFromCacheDataset(CACHE_PATH / "train", 12)
valid_ds = DirectTensorLoadFromCacheDataset(CACHE_PATH / "valid", 12)

In [ ]:
from mtrain.utils import it_chain

show(it_chain(GenericMaskClassificationDataset.denormalize(train_ds[0][0], 3)))


In [ ]:
dls = DataLoaders.from_dsets(train_ds, valid_ds, device=default_device())

## train model

In [ ]:
from mtrain.neg_mask.model.learner import load_our_learner

learn = vision_learner(
    dls,
    mobilenet_v3_large,
    n_in=12,
    metrics=[accuracy, F1Score(average="macro")],
    loss_func=CrossEntropyLossFlat(),
    n_out=len(labels),
    normalize=False,
)
learn = learn.remove_cb(ProgressCallback)
# state_dict = torch.load('/Users/hariomnarang/Desktop/personal/roads/datasets/models/taco_pretrained_mask_classifier/mobilenet_v3_large_130x130_iter-20-pure-torch.pth')
# keys_to_remove = [k for k in state_dict.keys() if '1.8' in k]
# print("Removing:", keys_to_remove)
# for k in keys_to_remove:
#     del state_dict[k]

# learn.model.load_state_dict(state_dict, strict=False)
# print("loaded")

In [ ]:
# learn.freeze()
learn.fine_tune(1)

In [ ]:
learn.fit_one_cycle(20)

In [ ]:
MODEL_OUT_DIR = mkdir(Path("../../datasets/models/taco_pretrained_mask_classifier"))

In [ ]:
learn.save((MODEL_OUT_DIR / "mobilenet_v3_large_12_chan_iter-40").resolve())

In [ ]:
from  mtrain.neg_mask.model.show import show_confusion_matrix
show_confusion_matrix(learn, dls.valid, LABELS)

In [ ]:
from mtrain.neg_mask.model.show import *

In [ ]:
all_preds, all_targs, decoded, all_losses = get_preds_for_ds(learn, valid_ds, 4)

In [ ]:
show_classification_report(all_preds, all_targs, LABELS)